# 06 — Annotate gold + Genie space (practice data)

Two parts, run after gold (notebook 05) is built:

**Part 1 — Annotate the gold tables.** Add `COMMENT`s on the gold tables and key columns. Genie (and AI/BI) read this Unity Catalog metadata to understand the data. Table and column comments move NL accuracy as much as the keys do.

**Part 2 — Build one Genie space over the practice star.** Created programmatically with the Databricks SDK (`w.genie.create_space`) from a `serialized_space` (v2) payload, so it is reproducible. Each piece lands in its slot in the Genie config:
- **General instructions** → `instructions.text_instructions`
- **Certified example Q&A** → `instructions.example_question_sqls`
- **Curated filters and measures** → `instructions.sql_snippets`
- **Sample questions** (UI chips) → `config.sample_questions`
- **Per-column descriptions** → `data_sources.tables[].column_configs`

**Joins.** Genie infers joins from three signals here: (1) the RELY PK/FK constraints declared in 05 (the primary signal), (2) the certified example SQL below (each encodes a practice join), and (3) an explicit join note in the text instructions. The structured `join_specs` field is not set programmatically, since its condition does not round-trip through the create API, but the above fully convey the relationships. You can also add visual joins in the Genie UI.

> **Run this notebook interactively** (it is not part of the pipeline Job). Set `UC_CATALOG`, `UC_SCHEMA`, and `SQL_WAREHOUSE_ID` in the widgets at the top when running in the workspace UI (they fall back to `.env` for local Databricks Connect runs). Prereqs: the gold practice tables from `05` exist (`fact_practice_event`, `dim_session`, plus shared `dim_player` / `dim_team` / `dim_date`) with RELY constraints.

> Scope: one Genie space over the practice data. No dashboard (out of scope).

In [ ]:
%pip install -q --upgrade "databricks-sdk>=0.117.0"
dbutils.library.restartPython()
# Workspace only: the Genie Spaces API (w.genie.create_space) needs a recent databricks-sdk. The cluster
# runtime bundles an older one, so upgrade it and restart Python to load it. Run this cell first, then the
# rest top-to-bottom. Local Databricks Connect already has it via requirements.txt, so skip this cell there.

In [ ]:
# Dual-mode setup: workspace UI (widgets) or local Databricks Connect (.env).
import os, json
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
try:
    spark  # already present inside a Databricks workspace notebook
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

# Config: in the workspace UI these show as editable text widgets at the top of the notebook (seeded from
# .env when present). Under local Databricks Connect (no dbutils) they fall back to .env / os.environ.
def cfg(name, default=""):
    try:
        dbutils.widgets.text(name, os.getenv(name, default))   # noqa: F821 — creates/seeds the UI widget
        return dbutils.widgets.get(name) or default
    except Exception:                                          # local Connect: no dbutils
        return os.getenv(name, default)

UC_CATALOG       = cfg("UC_CATALOG", "main")
UC_SCHEMA        = cfg("UC_SCHEMA", "synergy_baseball_demo")
SQL_WAREHOUSE_ID = cfg("SQL_WAREHOUSE_ID")
GOLD_SCHEMA = f"{UC_SCHEMA}_gold"
G = f"{UC_CATALOG}.{GOLD_SCHEMA}"
print("target gold:", G)
print("warehouse:", SQL_WAREHOUSE_ID or "not set: fill the SQL_WAREHOUSE_ID widget above (Part 2 needs it)")

## Part 1 — Annotate the gold tables

`COMMENT ON TABLE` / `COMMENT ON COLUMN` so Genie reads what each table and key column means. Table comments cover all 13 gold tables; column comments focus on the practice-space tables (keys, FKs, and the headline pitch/contact measures).

In [ ]:
# Table comments — all 13 gold tables.
TABLE_COMMENTS = {
    "dim_team":         "Team master (one row per team). Surrogate key team_key; carries mlbam_team_id, the MLBAM cross-source crosswalk.",
    "dim_player":       "Player master (one row per player). Surrogate key player_key; carries mlbam_player_id.",
    "dim_date":         "Conformed calendar, one row per date referenced by any fact. Key date_key as yyyyMMdd.",
    "dim_venue":        "Ballpark / venue master. Surrogate key venue_key.",
    "dim_competition":  "Competition / league grouping. Surrogate key competition_key.",
    "dim_umpire":       "Umpire master. Surrogate key umpire_key; carries mlbam_umpire_id.",
    "dim_session":      "Practice session master and parent of practice events. Surrogate key session_key; one row per practice / training session.",
    "fact_game":        "One row per game. Final score, total runs, run differential and winner are derived from the running score on events.",
    "fact_event":       "One row per in-game event (plate-appearance event). FKs to game, batter, teams, umpire and date.",
    "fact_pitch":       "One row per in-game pitch (pitch-level subset).",
    "fact_practice_event":   "One row per PRACTICE event (tracked pitch/swing in a practice or training session). Wide table with full pitch, contact and tracking metrics. FKs to dim_session, batter dim_player, dim_team and dim_date.",
    "fact_practice_workout": "One row per practice training-workout group. FKs to dim_session, dim_team and dim_date.",
    "fact_roster_stint":     "Player-team stints over time (from team history). FKs to dim_team and dim_date (start/end). No player FK - the source has no player id.",
}

for t, comment in TABLE_COMMENTS.items():
    spark.sql(f"COMMENT ON TABLE {G}.{t} IS '{comment}'")
print(f"annotated {len(TABLE_COMMENTS)} gold tables")

In [ ]:
# Column comments — focused on the practice-space tables (keys, FKs, headline measures).
COLUMN_COMMENTS = {
    "fact_practice_event": {
        "practice_event_key": "Surrogate primary key (md5 of the source practice-event id).",
        "session_key":        "FK to dim_session.session_key (the practice session this event belongs to).",
        "batter_key":         "FK to dim_player.player_key (the batter). There is no pitcher key.",
        "home_team_key":      "FK to dim_team.team_key (home/owning team of the session).",
        "away_team_key":      "FK to dim_team.team_key (visiting team, when present).",
        "date_key":           "FK to dim_date.date_key (yyyyMMdd integer).",
        "pitch_pitch_speed_mph":        "Pitch velocity out of hand, in mph.",
        "pitch_pitch_kind":             "Pitch type classification (e.g. fastball, slider).",
        "pitch_pitch_result":           "Outcome of the pitch (e.g. ball, called strike, in play).",
        "contact_exit_speed_mph":       "Batted-ball exit velocity, in mph.",
        "contact_launch_angle_degrees": "Batted-ball launch angle, in degrees.",
        "contact_distance_feet":        "Batted-ball distance, in feet.",
        "unofficial_event":             "True if the event is unofficial; filter to false for official activity.",
    },
    "fact_practice_workout": {
        "workout_key":  "Surrogate primary key (md5 of session_home_team_id + group_number).",
        "session_key":  "FK to dim_session.session_key.",
        "home_team_key":"FK to dim_team.team_key.",
        "date_key":     "FK to dim_date.date_key.",
        "group_number": "Training-workout group number within the session.",
    },
    "dim_session": {
        "session_key":          "Surrogate primary key.",
        "practice_session_type":"Type of practice / training session.",
        "season":               "Season year of the session.",
        "home_team_name":       "Home / owning team name.",
        "away_team_name":       "Visiting team name, when present.",
        "practice_date":        "Date of the session (string, yyyy-MM-dd).",
    },
    "dim_player": {
        "player_key":       "Surrogate primary key.",
        "mlbam_player_id":  "MLBAM player id - cross-source crosswalk key.",
        "player_name":      "Full player name.",
        "position":         "Primary position.",
        "bats_handedness":  "Batting handedness (L/R/S).",
        "throws_handedness":"Throwing handedness (L/R).",
    },
    "dim_team": {
        "team_key":     "Surrogate primary key.",
        "name":         "Team name.",
        "mlbam_team_id":"MLBAM team id - cross-source crosswalk key.",
    },
    "dim_date": {
        "date_key":   "Surrogate primary key, yyyyMMdd integer.",
        "full_date":  "Calendar date.",
        "year":       "Calendar year.",
        "month":      "Calendar month (1-12).",
        "month_name": "Full month name.",
    },
}

n = 0
for t, cols in COLUMN_COMMENTS.items():
    for col, comment in cols.items():
        spark.sql(f"COMMENT ON COLUMN {G}.{t}.{col} IS '{comment}'")
        n += 1
print(f"annotated {n} columns across {len(COLUMN_COMMENTS)} practice-space tables")

## Part 2 — Genie space over the practice star

Builds one Genie space around the practice data with the SDK. The `serialized_space` (v2) places each piece in its canonical slot: general instructions, certified example SQL (which encode the joins), curated filters/measures, sample questions, and per-column descriptions. Re-running updates the same space in place (matched by title). Prints the space URL at the end.

In [ ]:
import uuid
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
warehouse_id = SQL_WAREHOUSE_ID
assert warehouse_id, "SQL_WAREHOUSE_ID is not set: fill the SQL_WAREHOUSE_ID widget (workspace) or set it in .env (local)"
hid = lambda: uuid.uuid4().hex   # Genie ids must be lowercase 32-hex, no hyphens

# Per-column descriptions for the wide event fact (format/entity assistance).
# column_configs must be sorted by column_name (the API enforces it); sorted() handles that.
_event_cols = {
    "contact_distance_feet":        "Batted-ball distance in feet.",
    "contact_exit_speed_mph":       "Batted-ball exit velocity (exit speed) in mph.",
    "contact_launch_angle_degrees": "Batted-ball launch angle in degrees.",
    "pitch_pitch_kind":             "Pitch type classification (e.g. fastball, slider).",
    "pitch_pitch_result":           "Outcome of the pitch (e.g. ball, called strike, in play).",
    "pitch_pitch_speed_mph":        "Pitch velocity out of hand, in mph.",
}
event_col_configs = [{"column_name": c, "description": [d]} for c, d in sorted(_event_cols.items())]

FPE = f"{G}.fact_practice_event"
serialized_space = {
    "version": 2,
    # Sample questions shown as chips in the Genie UI
    "config": {"sample_questions": [{"id": hid(), "question": [q]} for q in [
        "How many practice events were recorded per session type?",
        "Which batters had the highest exit velocity in practice?",
        "What is the average pitch velocity by pitch kind?",
        "How many practice events happened each month?",
        "What is the average launch angle for batted balls in practice?",
    ]]},
    # Tables in the space (the practice star plus shared dimensions).
    # tables must be sorted by identifier (API enforces it); sorted() handles that.
    "data_sources": {"tables": sorted([
        {"identifier": FPE, "column_configs": event_col_configs},
        {"identifier": f"{G}.fact_practice_workout"},
        {"identifier": f"{G}.dim_session"},
        {"identifier": f"{G}.dim_player"},
        {"identifier": f"{G}.dim_team"},
        {"identifier": f"{G}.dim_date"},
    ], key=lambda t: t["identifier"])},
    "instructions": {
        # General instructions: scope, joins, caveats, units
        "text_instructions": [{"id": hid(), "content": ["\n".join([
            "This space answers questions about PRACTICE data ONLY (batting practice and training sessions), "
            "not games. fact_practice_event = one row per tracked practice event (pitch/swing); "
            "fact_practice_workout = one row per training-workout group.",
            "Joins: fact_practice_event.session_key = dim_session.session_key; "
            "fact_practice_event.batter_key = dim_player.player_key (the batter); "
            "fact_practice_event.home_team_key or away_team_key = dim_team.team_key; "
            "fact_practice_event.date_key = dim_date.date_key. "
            "fact_practice_workout.session_key = dim_session.session_key. "
            "These relationships are also declared as RELY foreign keys in the gold schema.",
            "There is no pitcher entity: the source has no pitcher id, so player-level analysis covers the "
            "batter only.",
            "Filter unofficial_event = false to count only official activity. Units: speeds are mph, angles "
            "are degrees, distances are feet.",
        ])]}],
        # Certified example Q&A: each demonstrates a practice join so Genie learns the relationships
        "example_question_sqls": [
            {"id": hid(), "question": ["Average exit speed by practice session type"], "sql": [
                f"SELECT s.practice_session_type, AVG(e.contact_exit_speed_mph) AS avg_exit_mph "
                f"FROM {FPE} e JOIN {G}.dim_session s ON e.session_key = s.session_key "
                f"WHERE e.contact_exit_speed_mph IS NOT NULL "
                f"GROUP BY s.practice_session_type ORDER BY avg_exit_mph DESC"]},
            {"id": hid(), "question": ["Top batters by max exit velocity in practice"], "sql": [
                f"SELECT p.player_name, MAX(e.contact_exit_speed_mph) AS max_exit_mph "
                f"FROM {FPE} e JOIN {G}.dim_player p ON e.batter_key = p.player_key "
                f"WHERE e.contact_exit_speed_mph IS NOT NULL "
                f"GROUP BY p.player_name ORDER BY max_exit_mph DESC LIMIT 10"]},
            {"id": hid(), "question": ["Practice events per month"], "sql": [
                f"SELECT d.year, d.month, COUNT(*) AS events "
                f"FROM {FPE} e JOIN {G}.dim_date d ON e.date_key = d.date_key "
                f"GROUP BY d.year, d.month ORDER BY d.year, d.month"]},
            {"id": hid(), "question": ["Average pitch velocity by pitch kind"], "sql": [
                f"SELECT pitch_pitch_kind, AVG(pitch_pitch_speed_mph) AS avg_mph, COUNT(*) AS pitches "
                f"FROM {FPE} WHERE pitch_pitch_speed_mph IS NOT NULL "
                f"GROUP BY pitch_pitch_kind ORDER BY pitches DESC"]},
        ],
        # Curated filters and measures (reusable building blocks)
        "sql_snippets": {
            "filters": [
                {"id": hid(), "display_name": "Batted balls only", "sql": ["contact_exit_speed_mph IS NOT NULL"]},
                {"id": hid(), "display_name": "Official events only", "sql": ["unofficial_event = false"]},
            ],
            "measures": [
                {"id": hid(), "display_name": "Avg exit speed (mph)",   "sql": ["AVG(contact_exit_speed_mph)"]},
                {"id": hid(), "display_name": "Max exit speed (mph)",   "sql": ["MAX(contact_exit_speed_mph)"]},
                {"id": hid(), "display_name": "Avg pitch velocity (mph)","sql": ["AVG(pitch_pitch_speed_mph)"]},
                {"id": hid(), "display_name": "Avg launch angle (deg)", "sql": ["AVG(contact_launch_angle_degrees)"]},
            ],
        },
    },
}

TITLE = f"[{UC_SCHEMA}] Synergy Practice (Genie)"
DESC  = "Natural-language Q&A over the Synergy practice star schema (practice events, workouts, sessions, players, teams, dates)."
def _normalize(sp):
    # The Genie API requires every repeated list in serialized_space to be sorted by its key
    # (tables by identifier, column_configs by column_name, all id-bearing lists by id).
    sp = json.loads(json.dumps(sp))  # deep copy
    sp.get("config", {}).get("sample_questions", []).sort(key=lambda x: x["id"])
    tables = sp.get("data_sources", {}).get("tables", [])
    tables.sort(key=lambda t: t["identifier"])
    for t in tables:
        t.get("column_configs", []).sort(key=lambda c: c["column_name"])
    instr = sp.get("instructions", {})
    for k in ("text_instructions", "example_question_sqls"):
        instr.get(k, []).sort(key=lambda x: x["id"])
    snip = instr.get("sql_snippets", {})
    for k in ("filters", "measures"):
        snip.get(k, []).sort(key=lambda x: x["id"])
    return sp

ser   = json.dumps(_normalize(serialized_space))

# Idempotent: update the space if one with this title already exists, else create it.
def _find_space_by_title(title):
    token = None
    while True:
        resp = w.genie.list_spaces(page_token=token)
        for s in (resp.spaces or []):
            if s.title == title:
                return s
        token = resp.next_page_token
        if not token:
            return None

existing = _find_space_by_title(TITLE)
if existing:
    w.genie.update_space(existing.space_id, serialized_space=ser, title=TITLE, description=DESC, warehouse_id=warehouse_id)
    space_id, action = existing.space_id, "updated"
else:
    sp = w.genie.create_space(warehouse_id=warehouse_id, serialized_space=ser, title=TITLE, description=DESC)
    space_id, action = sp.space_id, "created"

print(f"Genie space {action}: {space_id}")
print(f"Open: {w.config.host}/genie/rooms/{space_id}")